In [ ]:
!pip install datasets transformers accelerate evaluate -q

In [ ]:
# ============================================================
# Imports and Reproducibility Setup
# ============================================================

import os
import re
import copy
import random
import numpy as np
import pandas as pd
import torch

from collections import Counter

from datasets import load_dataset
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

SEED = 42

os.environ["PYTHONHASHSEED"] = str(SEED)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

device = "cuda" if torch.cuda.is_available() else "cpu"

print("Device:", device)
print("Seed:", SEED)

# Load DataSet

In [ ]:
dataset = load_dataset("imdb")

print(dataset)
print(dataset["train"][0])
print(dataset["test"][0])

train_full_df = pd.DataFrame(dataset["train"])
test_df = pd.DataFrame(dataset["test"])

train_full_df = train_full_df[["text", "label"]]
test_df = test_df[["text", "label"]]

print(train_full_df.head())
print(test_df.head())

SEED = 42

train_df, val_df = train_test_split(
    train_full_df,
    test_size=0.2,
    random_state=SEED,
    stratify=train_full_df["label"]
)

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print("Train size:", len(train_df))
print("Validation size:", len(val_df))
print("Test size:", len(test_df))

dataset_summary = pd.DataFrame({
    "Split": ["Train", "Validation", "Test"],
    "Number of examples": [len(train_df), len(val_df), len(test_df)],
    "Negative examples": [
        (train_df["label"] == 0).sum(),
        (val_df["label"] == 0).sum(),
        (test_df["label"] == 0).sum()
    ],
    "Positive examples": [
        (train_df["label"] == 1).sum(),
        (val_df["label"] == 1).sum(),
        (test_df["label"] == 1).sum()
    ]
})

dataset_summary



# TF-IDF + Logistic Regression:

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
import pandas as pd
import numpy as np

# Data
X_train = train_df["text"]
y_train = train_df["label"]

X_val = val_df["text"]
y_val = val_df["label"]

# TF-IDF vectorizer
tfidf_vectorizer = TfidfVectorizer(
    lowercase=True,
    stop_words=None,
    max_features=30000,
    ngram_range=(1, 2),
    min_df=2
)

# Convert text into sparse TF-IDF vectors
X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)
X_val_tfidf = tfidf_vectorizer.transform(X_val)

print("Train TF-IDF shape:", X_train_tfidf.shape)
print("Validation TF-IDF shape:", X_val_tfidf.shape)

# Logistic Regression classifier
tfidf_lr_model = LogisticRegression(
    max_iter=1000,
    random_state=SEED,
    solver="liblinear"
)

# Train
tfidf_lr_model.fit(X_train_tfidf, y_train)

# Validate
y_val_pred_tfidf = tfidf_lr_model.predict(X_val_tfidf)

# Metrics
val_acc_tfidf = accuracy_score(y_val, y_val_pred_tfidf)
val_f1_tfidf = f1_score(y_val, y_val_pred_tfidf, average="macro")

print("TF-IDF + Logistic Regression Validation Results")
print("Accuracy:", val_acc_tfidf)
print("Macro-F1:", val_f1_tfidf)

print("\nClassification Report:")
print(classification_report(y_val, y_val_pred_tfidf, target_names=["negative", "positive"]))

# TF-IDF + Logistic Regression with English stopword removal

In [ ]:
# TF-IDF + Logistic Regression with stopword removal

tfidf_stopword_vectorizer = TfidfVectorizer(
    lowercase=True,
    stop_words="english",
    max_features=30000,
    ngram_range=(1, 2),
    min_df=2
)

X_train_tfidf_stop = tfidf_stopword_vectorizer.fit_transform(X_train)
X_val_tfidf_stop = tfidf_stopword_vectorizer.transform(X_val)

print("Train TF-IDF with stopword removal shape:", X_train_tfidf_stop.shape)
print("Validation TF-IDF with stopword removal shape:", X_val_tfidf_stop.shape)

tfidf_lr_stop_model = LogisticRegression(
    max_iter=1000,
    random_state=SEED,
    solver="liblinear"
)

tfidf_lr_stop_model.fit(X_train_tfidf_stop, y_train)

y_val_pred_tfidf_stop = tfidf_lr_stop_model.predict(X_val_tfidf_stop)

val_acc_tfidf_stop = accuracy_score(y_val, y_val_pred_tfidf_stop)
val_f1_tfidf_stop = f1_score(y_val, y_val_pred_tfidf_stop, average="macro")

print("TF-IDF + Logistic Regression with Stopword Removal Validation Results")
print("Accuracy:", val_acc_tfidf_stop)
print("Macro-F1:", val_f1_tfidf_stop)

print("\nClassification Report:")
print(classification_report(y_val, y_val_pred_tfidf_stop, target_names=["negative", "positive"]))

# TF-IDF + Logistic Regression with truncation (first 256 words):

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, classification_report

# Helper function: keep only the first N words of each review
def truncate_words(text, max_words=256):
    words = text.split()
    return " ".join(words[:max_words])

# Apply truncation to train and validation text
X_train_trunc_256 = train_df["text"].apply(lambda x: truncate_words(x, max_words=256))
X_val_trunc_256 = val_df["text"].apply(lambda x: truncate_words(x, max_words=256))

y_train = train_df["label"]
y_val = val_df["label"]

# TF-IDF vectorizer for truncated text
tfidf_trunc_vectorizer = TfidfVectorizer(
    lowercase=True,
    stop_words=None,
    max_features=30000,
    ngram_range=(1, 2),
    min_df=2
)

# Learn vocabulary from truncated training reviews
X_train_tfidf_trunc = tfidf_trunc_vectorizer.fit_transform(X_train_trunc_256)

# Transform truncated validation reviews using the same vocabulary
X_val_tfidf_trunc = tfidf_trunc_vectorizer.transform(X_val_trunc_256)

print("Train truncated TF-IDF shape:", X_train_tfidf_trunc.shape)
print("Validation truncated TF-IDF shape:", X_val_tfidf_trunc.shape)

# Logistic Regression classifier
tfidf_lr_trunc_model = LogisticRegression(
    max_iter=1000,
    random_state=SEED,
    solver="liblinear"
)

# Train
tfidf_lr_trunc_model.fit(X_train_tfidf_trunc, y_train)

# Predict on validation set
y_val_pred_tfidf_trunc = tfidf_lr_trunc_model.predict(X_val_tfidf_trunc)

# Evaluate
val_acc_tfidf_trunc = accuracy_score(y_val, y_val_pred_tfidf_trunc)
val_f1_tfidf_trunc = f1_score(y_val, y_val_pred_tfidf_trunc, average="macro")

print("TF-IDF + Logistic Regression with 256-word Truncation Validation Results")
print("Accuracy:", val_acc_tfidf_trunc)
print("Macro-F1:", val_f1_tfidf_trunc)

print("\nClassification Report:")
print(classification_report(y_val, y_val_pred_tfidf_trunc, target_names=["negative", "positive"]))

# Neural Dense Representation Model(BiLSTM):

In [ ]:
import re
import copy
import random
import numpy as np
import pandas as pd
from collections import Counter

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.metrics import accuracy_score, f1_score, classification_report

# -----------------------------
# 1. Reproducibility
# -----------------------------

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

# -----------------------------
# 2. Better tokenizer
# -----------------------------
# This keeps contractions like "wasn't", "don't", "it's" together.

def simple_tokenize(text):
    text = text.lower()
    tokens = re.findall(r"[a-z]+(?:'[a-z]+)?|\d+|[^\w\s]", text)
    return tokens

# -----------------------------
# 3. Build vocabulary from training set only
# -----------------------------

MAX_VOCAB_SIZE = 30000
MIN_FREQ = 2

counter = Counter()

for text in train_df["text"]:
    counter.update(simple_tokenize(text))

PAD_TOKEN = "<PAD>"
UNK_TOKEN = "<UNK>"

vocab = {
    PAD_TOKEN: 0,
    UNK_TOKEN: 1
}

for token, freq in counter.most_common(MAX_VOCAB_SIZE - 2):
    if freq >= MIN_FREQ:
        vocab[token] = len(vocab)

print("Vocabulary size:", len(vocab))

# -----------------------------
# 4. Encoding with lengths
# -----------------------------

MAX_LEN = 256

def encode_text(text, vocab, max_len=MAX_LEN):
    tokens = simple_tokenize(text)
    token_ids = [vocab.get(token, vocab[UNK_TOKEN]) for token in tokens]

    token_ids = token_ids[:max_len]
    length = len(token_ids)

    if length == 0:
        token_ids = [vocab[UNK_TOKEN]]
        length = 1

    padding_length = max_len - len(token_ids)
    token_ids = token_ids + [vocab[PAD_TOKEN]] * padding_length

    return token_ids, length

class IMDbDataset(Dataset):
    def __init__(self, texts, labels, vocab, max_len=MAX_LEN):
        self.texts = texts.tolist()
        self.labels = labels.tolist()
        self.vocab = vocab
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        input_ids, length = encode_text(self.texts[idx], self.vocab, self.max_len)
        label = self.labels[idx]

        return {
            "input_ids": torch.tensor(input_ids, dtype=torch.long),
            "length": torch.tensor(length, dtype=torch.long),
            "label": torch.tensor(label, dtype=torch.long)
        }

train_dataset = IMDbDataset(train_df["text"], train_df["label"], vocab, MAX_LEN)
val_dataset = IMDbDataset(val_df["text"], val_df["label"], vocab, MAX_LEN)

BATCH_SIZE = 64

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

# -----------------------------
# 5. Packed BiLSTM classifier
# -----------------------------

class PackedBiLSTMClassifier(nn.Module):
    def __init__(
        self,
        vocab_size,
        embedding_dim=128,
        hidden_dim=128,
        num_layers=1,
        num_classes=2,
        dropout=0.5,
        pad_idx=0
    ):
        super(PackedBiLSTMClassifier, self).__init__()

        self.embedding = nn.Embedding(
            num_embeddings=vocab_size,
            embedding_dim=embedding_dim,
            padding_idx=pad_idx
        )

        self.lstm = nn.LSTM(
            input_size=embedding_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=True
        )

        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(hidden_dim * 2, num_classes)

    def forward(self, input_ids, lengths):
        embedded = self.embedding(input_ids)

        # Pack padded sequences so the LSTM ignores <PAD> positions
        packed_embedded = nn.utils.rnn.pack_padded_sequence(
            embedded,
            lengths.cpu(),
            batch_first=True,
            enforce_sorted=False
        )

        packed_output, (hidden, cell) = self.lstm(packed_embedded)

        forward_hidden = hidden[-2]
        backward_hidden = hidden[-1]

        final_hidden = torch.cat((forward_hidden, backward_hidden), dim=1)
        final_hidden = self.dropout(final_hidden)

        logits = self.classifier(final_hidden)

        return logits

model = PackedBiLSTMClassifier(
    vocab_size=len(vocab),
    embedding_dim=128,
    hidden_dim=128,
    num_layers=1,
    num_classes=2,
    dropout=0.5,
    pad_idx=vocab[PAD_TOKEN]
).to(device)

print(model)

# -----------------------------
# 6. Training setup
# -----------------------------

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=5e-4,
    weight_decay=1e-4
)

EPOCHS = 8
PATIENCE = 2

# -----------------------------
# 7. Training / evaluation functions
# -----------------------------

def train_one_epoch(model, data_loader, optimizer, criterion, device):
    model.train()
    total_loss = 0

    for batch in data_loader:
        input_ids = batch["input_ids"].to(device)
        lengths = batch["length"].to(device)
        labels = batch["label"].to(device)

        optimizer.zero_grad()

        logits = model(input_ids, lengths)
        loss = criterion(logits, labels)

        loss.backward()

        # Helps avoid exploding gradients in RNNs
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(data_loader)


def evaluate(model, data_loader, criterion, device):
    model.eval()

    total_loss = 0
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for batch in data_loader:
            input_ids = batch["input_ids"].to(device)
            lengths = batch["length"].to(device)
            labels = batch["label"].to(device)

            logits = model(input_ids, lengths)
            loss = criterion(logits, labels)

            total_loss += loss.item()

            preds = torch.argmax(logits, dim=1)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    avg_loss = total_loss / len(data_loader)
    accuracy = accuracy_score(all_labels, all_preds)
    macro_f1 = f1_score(all_labels, all_preds, average="macro")

    return avg_loss, accuracy, macro_f1, all_labels, all_preds

# -----------------------------
# 8. Train with early stopping
# -----------------------------

best_val_f1 = 0
best_model_state = None
epochs_without_improvement = 0

for epoch in range(EPOCHS):
    train_loss = train_one_epoch(model, train_loader, optimizer, criterion, device)

    val_loss, val_acc, val_f1, val_labels, val_preds = evaluate(
        model,
        val_loader,
        criterion,
        device
    )

    print(f"Epoch {epoch + 1}/{EPOCHS}")
    print(f"Train loss: {train_loss:.4f}")
    print(f"Validation loss: {val_loss:.4f}")
    print(f"Validation accuracy: {val_acc:.4f}")
    print(f"Validation Macro-F1: {val_f1:.4f}")
    print("-" * 50)

    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        best_model_state = copy.deepcopy(model.state_dict())
        epochs_without_improvement = 0
    else:
        epochs_without_improvement += 1

    if epochs_without_improvement >= PATIENCE:
        print("Early stopping triggered.")
        break

model.load_state_dict(best_model_state)

val_loss, val_acc_bilstm, val_f1_bilstm, val_labels_bilstm, val_preds_bilstm = evaluate(
    model,
    val_loader,
    criterion,
    device
)

print("Best Packed BiLSTM Validation Results")
print("Accuracy:", val_acc_bilstm)
print("Macro-F1:", val_f1_bilstm)

print("\nClassification Report:")
print(classification_report(val_labels_bilstm, val_preds_bilstm, target_names=["negative", "positive"]))

# Transformer-based model DistilBERT:

In [ ]:
import random
import numpy as np
import pandas as pd
import torch

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding
)

from sklearn.metrics import accuracy_score, f1_score, classification_report

reproducibility setup:

In [ ]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

In [ ]:
# Keep only text and label columns
train_hf = Dataset.from_pandas(train_df[["text", "label"]])
val_hf = Dataset.from_pandas(val_df[["text", "label"]])

print(train_hf)
print(val_hf)
print(train_hf[0])

load tokenizer:

In [ ]:
MODEL_NAME = "distilbert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

tokenize train and validation data:

In [ ]:
MAX_LEN = 256

def tokenize_batch(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        max_length=MAX_LEN
    )

train_tokenized = train_hf.map(tokenize_batch, batched=True)
val_tokenized = val_hf.map(tokenize_batch, batched=True)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

print(train_tokenized[0].keys())
print(train_tokenized[0])

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=1)

    acc = accuracy_score(labels, predictions)
    macro_f1 = f1_score(labels, predictions, average="macro")

    return {
        "accuracy": acc,
        "macro_f1": macro_f1
    }

In [ ]:
model_distilbert = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2
)

training arguments:

In [ ]:
training_args = TrainingArguments(
    output_dir="./distilbert_imdb_results",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=2,
    weight_decay=0.01,
    logging_dir="./distilbert_logs",
    logging_steps=100,
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,
    seed=SEED,
    report_to="none"
)

In [ ]:
trainer = Trainer(
    model=model_distilbert,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=val_tokenized,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

train DistilBERT:

In [ ]:
trainer.train()

evaluate on validation set:

In [ ]:
distilbert_val_results = trainer.evaluate()

print(distilbert_val_results)

In [ ]:
pred_output = trainer.predict(val_tokenized)

logits = pred_output.predictions
y_val_pred_distilbert = np.argmax(logits, axis=1)
y_val_true_distilbert = pred_output.label_ids

val_acc_distilbert = accuracy_score(y_val_true_distilbert, y_val_pred_distilbert)
val_f1_distilbert = f1_score(y_val_true_distilbert, y_val_pred_distilbert, average="macro")

print("DistilBERT Validation Results")
print("Accuracy:", val_acc_distilbert)
print("Macro-F1:", val_f1_distilbert)

print("\nClassification Report:")
print(classification_report(
    y_val_true_distilbert,
    y_val_pred_distilbert,
    target_names=["negative", "positive"]
))

# All validation results:

In [ ]:
validation_results = pd.DataFrame([
    {
        "Model": "TF-IDF + Logistic Regression",
        "Representation": "Sparse lexical",
        "Tokenization": "Word/ngram TF-IDF",
        "Preprocessing": "Lowercase, no stopword removal, full text",
        "Validation Accuracy": val_acc_tfidf,
        "Validation Macro-F1": val_f1_tfidf
    },
    {
        "Model": "TF-IDF + Logistic Regression",
        "Representation": "Sparse lexical",
        "Tokenization": "Word/ngram TF-IDF",
        "Preprocessing": "Lowercase, English stopword removal, full text",
        "Validation Accuracy": val_acc_tfidf_stop,
        "Validation Macro-F1": val_f1_tfidf_stop
    },
    {
        "Model": "TF-IDF + Logistic Regression",
        "Representation": "Sparse lexical",
        "Tokenization": "Word/ngram TF-IDF",
        "Preprocessing": "Lowercase, no stopword removal, first 256 words",
        "Validation Accuracy": val_acc_tfidf_trunc,
        "Validation Macro-F1": val_f1_tfidf_trunc
    },
    {
        "Model": "Packed BiLSTM",
        "Representation": "Dense learned",
        "Tokenization": "Word-level tokenizer",
        "Preprocessing": "Lowercase, vocabulary from train split, first 256 tokens",
        "Validation Accuracy": val_acc_bilstm,
        "Validation Macro-F1": val_f1_bilstm
    },
    {
        "Model": "DistilBERT",
        "Representation": "Contextual transformer",
        "Tokenization": "WordPiece/subword tokenizer",
        "Preprocessing": "DistilBERT tokenizer, first 256 subword tokens",
        "Validation Accuracy": val_acc_distilbert,
        "Validation Macro-F1": val_f1_distilbert
    }
])

validation_results

In [ ]:
validation_results.to_csv("q1_validation_results.csv", index=False)

# Model Comparison:

In [ ]:
main_model_results = pd.DataFrame([
    {
        "Model": "TF-IDF + Logistic Regression",
        "Representation": "Sparse lexical",
        "Validation Accuracy": val_acc_tfidf,
        "Validation Macro-F1": val_f1_tfidf
    },
    {
        "Model": "Packed BiLSTM",
        "Representation": "Dense learned",
        "Validation Accuracy": val_acc_bilstm,
        "Validation Macro-F1": val_f1_bilstm
    },
    {
        "Model": "DistilBERT",
        "Representation": "Contextual transformer",
        "Validation Accuracy": val_acc_distilbert,
        "Validation Macro-F1": val_f1_distilbert
    }
])

main_model_results

In [ ]:
main_model_results.to_csv("q1_main_model_results.csv", index=False)

# Misclassification Analysis for Each Main Model

TF-IDF misclassified examples:

In [ ]:
label_names = {
    0: "negative",
    1: "positive"
}

# Build TF-IDF error dataframe
tfidf_errors = val_df.copy()
tfidf_errors["true_label"] = y_val.values
tfidf_errors["predicted_label"] = y_val_pred_tfidf

tfidf_errors["true_label_name"] = tfidf_errors["true_label"].map(label_names)
tfidf_errors["predicted_label_name"] = tfidf_errors["predicted_label"].map(label_names)

# Optional: prediction confidence if available
tfidf_probs = tfidf_lr_model.predict_proba(X_val_tfidf)
tfidf_errors["predicted_confidence"] = tfidf_probs.max(axis=1)

# Keep only misclassified examples
tfidf_errors = tfidf_errors[
    tfidf_errors["true_label"] != tfidf_errors["predicted_label"]
].copy()

# Sort by confidence so we inspect strong but wrong predictions
tfidf_errors = tfidf_errors.sort_values(
    by="predicted_confidence",
    ascending=False
).reset_index(drop=True)

print("Number of TF-IDF validation errors:", len(tfidf_errors))

tfidf_errors[[
    "text",
    "true_label_name",
    "predicted_label_name",
    "predicted_confidence"
]].head(5)

In [ ]:
print("TF-IDF Misclassified Examples")
for i, row in tfidf_errors.head(5).iterrows():
    print("=" * 100)
    print("Example:", i + 1)
    print("Text:")
    print(row["text"])
    print("\nTrue label:", row["true_label_name"])
    print("Predicted label:", row["predicted_label_name"])
    print("Prediction confidence:", row["predicted_confidence"])

In [ ]:
tfidf_errors.head(20)[[
    "text",
    "true_label_name",
    "predicted_label_name",
    "predicted_confidence"
]].to_csv("q1_tfidf_misclassified_examples.csv", index=False)

BiLSTM misclassified examples:

In [ ]:
def get_bilstm_predictions_with_probs(model, data_loader, device):
    model.eval()

    all_preds = []
    all_probs = []
    all_labels = []

    with torch.no_grad():
        for batch in data_loader:
            input_ids = batch["input_ids"].to(device)
            lengths = batch["length"].to(device)
            labels = batch["label"].to(device)

            logits = model(input_ids, lengths)
            probs = torch.softmax(logits, dim=1)
            preds = torch.argmax(probs, dim=1)

            all_preds.extend(preds.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    return np.array(all_labels), np.array(all_preds), np.array(all_probs)


bilstm_true_labels, bilstm_preds, bilstm_probs = get_bilstm_predictions_with_probs(
    model,
    val_loader,
    device
)

# Check consistency with previous BiLSTM result
print("BiLSTM validation accuracy check:", accuracy_score(bilstm_true_labels, bilstm_preds))
print("BiLSTM validation Macro-F1 check:", f1_score(bilstm_true_labels, bilstm_preds, average="macro"))

In [ ]:
bilstm_errors = val_df.copy()
bilstm_errors["true_label"] = bilstm_true_labels
bilstm_errors["predicted_label"] = bilstm_preds

bilstm_errors["true_label_name"] = bilstm_errors["true_label"].map(label_names)
bilstm_errors["predicted_label_name"] = bilstm_errors["predicted_label"].map(label_names)
bilstm_errors["predicted_confidence"] = bilstm_probs.max(axis=1)

bilstm_errors = bilstm_errors[
    bilstm_errors["true_label"] != bilstm_errors["predicted_label"]
].copy()

bilstm_errors = bilstm_errors.sort_values(
    by="predicted_confidence",
    ascending=False
).reset_index(drop=True)

print("Number of BiLSTM validation errors:", len(bilstm_errors))

bilstm_errors[[
    "text",
    "true_label_name",
    "predicted_label_name",
    "predicted_confidence"
]].head(5)

In [ ]:
print("BiLSTM Misclassified Examples")
for i, row in bilstm_errors.head(5).iterrows():
    print("=" * 100)
    print("Example:", i + 1)
    print("Text:")
    print(row["text"])
    print("\nTrue label:", row["true_label_name"])
    print("Predicted label:", row["predicted_label_name"])
    print("Prediction confidence:", row["predicted_confidence"])

In [ ]:
bilstm_errors.head(20)[[
    "text",
    "true_label_name",
    "predicted_label_name",
    "predicted_confidence"
]].to_csv("q1_bilstm_misclassified_examples.csv", index=False)

DistilBERT misclassified examples:

In [ ]:
distilbert_probs = torch.softmax(
    torch.tensor(logits),
    dim=1
).numpy()

distilbert_errors = val_df.copy()
distilbert_errors["true_label"] = y_val_true_distilbert
distilbert_errors["predicted_label"] = y_val_pred_distilbert

distilbert_errors["true_label_name"] = distilbert_errors["true_label"].map(label_names)
distilbert_errors["predicted_label_name"] = distilbert_errors["predicted_label"].map(label_names)
distilbert_errors["predicted_confidence"] = distilbert_probs.max(axis=1)

distilbert_errors = distilbert_errors[
    distilbert_errors["true_label"] != distilbert_errors["predicted_label"]
].copy()

distilbert_errors = distilbert_errors.sort_values(
    by="predicted_confidence",
    ascending=False
).reset_index(drop=True)

print("Number of DistilBERT validation errors:", len(distilbert_errors))

distilbert_errors[[
    "text",
    "true_label_name",
    "predicted_label_name",
    "predicted_confidence"
]].head(5)

In [ ]:
print("DistilBERT Misclassified Examples")
for i, row in distilbert_errors.head(5).iterrows():
    print("=" * 100)
    print("Example:", i + 1)
    print("Text:")
    print(row["text"])
    print("\nTrue label:", row["true_label_name"])
    print("Predicted label:", row["predicted_label_name"])
    print("Prediction confidence:", row["predicted_confidence"])

In [ ]:
distilbert_errors.head(20)[[
    "text",
    "true_label_name",
    "predicted_label_name",
    "predicted_confidence"
]].to_csv("q1_distilbert_misclassified_examples.csv", index=False)

In [ ]:
tfidf_selected = tfidf_errors.head(5).copy()
tfidf_selected["model"] = "TF-IDF + Logistic Regression"

bilstm_selected = bilstm_errors.head(5).copy()
bilstm_selected["model"] = "Packed BiLSTM"

distilbert_selected = distilbert_errors.head(5).copy()
distilbert_selected["model"] = "DistilBERT"

selected_errors_all_models = pd.concat(
    [tfidf_selected, bilstm_selected, distilbert_selected],
    axis=0
).reset_index(drop=True)

selected_errors_all_models = selected_errors_all_models[[
    "model",
    "text",
    "true_label_name",
    "predicted_label_name",
    "predicted_confidence"
]]

selected_errors_all_models.to_csv(
    "q1_selected_misclassified_examples_all_models.csv",
    index=False
)

selected_errors_all_models

# Test Evaluation

TF-IDF:

In [ ]:
# =============================
# Final Test Evaluation: TF-IDF
# =============================

X_test = test_df["text"]
y_test = test_df["label"]

# Use the already-fitted TF-IDF vectorizer from the best validation configuration
X_test_tfidf = tfidf_vectorizer.transform(X_test)

# Predict test labels
y_test_pred_tfidf = tfidf_lr_model.predict(X_test_tfidf)

# Compute metrics
test_acc_tfidf = accuracy_score(y_test, y_test_pred_tfidf)
test_f1_tfidf = f1_score(y_test, y_test_pred_tfidf, average="macro")

print("TF-IDF + Logistic Regression Test Results")
print("Accuracy:", test_acc_tfidf)
print("Macro-F1:", test_f1_tfidf)

print("\nClassification Report:")
print(classification_report(
    y_test,
    y_test_pred_tfidf,
    target_names=["negative", "positive"]
))

BiLSTM:

In [ ]:
# =============================
# Final Test Evaluation: BiLSTM
# =============================

test_dataset = IMDbDataset(
    test_df["text"],
    test_df["label"],
    vocab,
    MAX_LEN
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

test_loss_bilstm, test_acc_bilstm, test_f1_bilstm, test_labels_bilstm, test_preds_bilstm = evaluate(
    model,
    test_loader,
    criterion,
    device
)

print("Packed BiLSTM Test Results")
print("Test loss:", test_loss_bilstm)
print("Accuracy:", test_acc_bilstm)
print("Macro-F1:", test_f1_bilstm)

print("\nClassification Report:")
print(classification_report(
    test_labels_bilstm,
    test_preds_bilstm,
    target_names=["negative", "positive"]
))

DistilBERT:

In [ ]:
# ================================
# Final Test Evaluation: DistilBERT
# ================================

test_hf = Dataset.from_pandas(test_df[["text", "label"]])

test_tokenized = test_hf.map(tokenize_batch, batched=True)

distilbert_test_output = trainer.predict(test_tokenized)

test_logits_distilbert = distilbert_test_output.predictions
y_test_pred_distilbert = np.argmax(test_logits_distilbert, axis=1)
y_test_true_distilbert = distilbert_test_output.label_ids

test_acc_distilbert = accuracy_score(
    y_test_true_distilbert,
    y_test_pred_distilbert
)

test_f1_distilbert = f1_score(
    y_test_true_distilbert,
    y_test_pred_distilbert,
    average="macro"
)

print("DistilBERT Test Results")
print("Accuracy:", test_acc_distilbert)
print("Macro-F1:", test_f1_distilbert)

print("\nClassification Report:")
print(classification_report(
    y_test_true_distilbert,
    y_test_pred_distilbert,
    target_names=["negative", "positive"]
))

In [ ]:
# =============================
# Final Test Results Table
# =============================

test_results = pd.DataFrame([
    {
        "Model": "TF-IDF + Logistic Regression",
        "Test Accuracy": test_acc_tfidf,
        "Test Macro-F1": test_f1_tfidf
    },
    {
        "Model": "Packed BiLSTM",
        "Test Accuracy": test_acc_bilstm,
        "Test Macro-F1": test_f1_bilstm
    },
    {
        "Model": "DistilBERT",
        "Test Accuracy": test_acc_distilbert,
        "Test Macro-F1": test_f1_distilbert
    }
])

test_results
test_results.to_csv("q1_final_test_results.csv", index=False)